In [1]:
# Model 4 — Admission Demand Forecasting
# This model aggregates admissions by month and forecasts future monthly demand.

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ==========================================================
# 1. LOAD DATA
# ==========================================================

file_path = "../data/processed/admissions/admissions_clean.csv"

df = pd.read_csv(file_path)

df["admission_date"] = pd.to_datetime(df["admission_date"])


# ==========================================================
# 2. CREATE MONTHLY ADMISSION COUNTS
# ==========================================================

monthly = (
    df.set_index("admission_date")
      .resample("MS")
      .size()
      .reset_index(name="admission_count")
)

print("Monthly dataset:")
print(monthly)

print("\nNumber of months:", len(monthly))


# ==========================================================
# 3. CREATE TIME FEATURES
# ==========================================================

monthly["month"] = monthly["admission_date"].dt.month
monthly["year"] = monthly["admission_date"].dt.year

monthly["time_index"] = np.arange(len(monthly))


# ==========================================================
# 4. CREATE LAG FEATURES
# ==========================================================

monthly["lag_1"] = monthly["admission_count"].shift(1)
monthly["lag_2"] = monthly["admission_count"].shift(2)
monthly["lag_3"] = monthly["admission_count"].shift(3)

monthly = monthly.dropna().reset_index(drop=True)


# ==========================================================
# 5. FEATURES AND TARGET
# ==========================================================

features = [
    "time_index",
    "month",
    "year",
    "lag_1",
    "lag_2",
    "lag_3"
]

X = monthly[features]
y = monthly["admission_count"]


# ==========================================================
# 6. TIME-BASED TRAIN / TEST SPLIT
# ==========================================================
# First 80% = training
# Last 20% = testing
# We do NOT randomly shuffle time-series data.

split = int(len(monthly) * 0.80)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# ==========================================================
# 7. TRAIN MODEL
# ==========================================================

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Admission Demand model...")

model.fit(X_train, y_train)

print("Training completed successfully!")


# ==========================================================
# 8. PREDICTIONS
# ==========================================================

y_pred = model.predict(X_test)

print("Predictions generated successfully!")


# ==========================================================
# 9. EVALUATION
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n========== ADMISSION DEMAND MODEL PERFORMANCE ==========")

print(f"MAE  : {mae:.2f} admissions")
print(f"RMSE : {rmse:.2f} admissions")


# ==========================================================
# 10. ACTUAL VS PREDICTED
# ==========================================================

results = pd.DataFrame({
    "Date": monthly.iloc[split:]["admission_date"].values,
    "Actual": y_test.values,
    "Predicted": np.round(y_pred, 0)
})

print("\n========== FORECAST RESULTS ==========")
print(results.to_string(index=False))

Monthly dataset:
   admission_date  admission_count
0      2023-01-01             1601
1      2023-02-01             1425
2      2023-03-01             1566
3      2023-04-01             1420
4      2023-05-01             1321
5      2023-06-01             1133
6      2023-07-01              996
7      2023-08-01             1057
8      2023-09-01             1176
9      2023-10-01             1568
10     2023-11-01             1594
11     2023-12-01             1583
12     2024-01-01             1658
13     2024-02-01             1424
14     2024-03-01             1652
15     2024-04-01             1490
16     2024-05-01             1322
17     2024-06-01             1106
18     2024-07-01             1005
19     2024-08-01             1104
20     2024-09-01             1290
21     2024-10-01             1497
22     2024-11-01             1683
23     2024-12-01             1676
24     2025-01-01             1621
25     2025-02-01             1442
26     2025-03-01             1571
27 

In [2]:
# Display only the forecasting evaluation metrics.

print("========== ADMISSION DEMAND MODEL PERFORMANCE ==========")
print(f"MAE  : {mae:.2f} admissions")
print(f"RMSE : {rmse:.2f} admissions")

========== ADMISSION DEMAND MODEL PERFORMANCE ==========
MAE  : 126.79 admissions
RMSE : 162.61 admissions


In [3]:
# Save the trained Admission Demand Forecasting model.

import joblib

model_path = "../ml/models/admission_demand_forecasting_model.pkl"

joblib.dump(model, model_path)

print("Admission Demand Forecasting model saved successfully!")
print("Saved at:", model_path)

Admission Demand Forecasting model saved successfully!
Saved at: ../ml/models/admission_demand_forecasting_model.pkl
